# ⚡ Sorting Algorithms: The Definitive Masterclass

A comprehensive, interactive, and rigorous deep dive into the **8 fundamental sorting algorithms in Computer Science**, their internal mechanics, asymptotic Big-O complexity proofs, stability guarantees, memory footprints, and practical benchmarks in Python.

---

## 🧭 Overview of the 8 Algorithms

| Algorithm | Core Mechanism | Time Complexity (Avg) | Space Complexity | Stability | In-Place? |
| :--- | :--- | :---: | :---: | :---: | :---: |
| **Bubble Sort** | Swaps adjacent out-of-order elements | $O(n^2)$ | $O(1)$ | ✅ Stable | ✅ Yes |
| **Selection Sort** | Selects the minimum element and moves it to the front | $O(n^2)$ | $O(1)$ | ❌ Unstable | ✅ Yes |
| **Insertion Sort** | Inserts each element into its sorted place in the sorted prefix | $O(n^2)$ | $O(1)$ | ✅ Stable | ✅ Yes |
| **Heap Sort** | Builds a Max-Heap and repeatedly extracts the root maximum | $O(n \log n)$ | $O(1)$ | ❌ Unstable | ✅ Yes |
| **Merge Sort** | Splits into halves, recursively sorts, and merges | $O(n \log n)$ | $O(n)$ | ✅ Stable | ❌ No |
| **Quick Sort** | Partitions around a pivot, then sorts partitions recursively | $O(n \log n)$ | $O(\log n)$ | ❌ Unstable | ✅ Yes |
| **Counting Sort** | Counts key frequencies and uses prefix sums for direct indexing | $O(n + k)$ | $O(n + k)$ | ✅ Stable | ❌ No |
| **Radix Sort** | Sorts digit by digit with stable counting passes (LSD) | $O(d \cdot (n + k))$ | $O(n + k)$ | ✅ Stable | ❌ No |

---

## 📑 Table of Contents
1. [Foundational Concepts: Stability, In-Place & Complexity Limits](#1.-Foundational-Concepts)
2. [Environment Setup & Visualization Utilities](#2.-Environment-Setup-&-Visualization-Utilities)
3. [Elementary Quadratic Algorithms ($O(n^2)$)](#3.-Elementary-Quadratic-Algorithms)
   - 3.1 Bubble Sort (with Early-Exit Optimization)
   - 3.2 Selection Sort
   - 3.3 Insertion Sort (and why it underpins Timsort)
4. [Efficient Divide-and-Conquer & Heap Algorithms ($O(n \log n)$)](#4.-Efficient-Algorithms)
   - 4.1 Heap Sort (Max-Heap Construction & In-Place Heapify)
   - 4.2 Merge Sort (Top-Down Divide & Conquer)
   - 4.3 Quick Sort (Randomized Pivot Partitioning)
5. [Non-Comparison Linear Time Algorithms](#5.-Non-Comparison-Linear-Time-Algorithms)
   - 5.1 Counting Sort (with negative integer support)
   - 5.2 Radix Sort (LSD - Least Significant Digit)
6. [Benchmarking Lab & Real-World Performance Analysis](#6.-Benchmarking-Lab)
7. [Master Complexity Matrix & Production Decision Guide](#7.-Master-Complexity-Matrix)

## 1. Foundational Concepts

Before diving into the algorithms, let us establish 3 crucial theoretical pillars:

### 1.1 Stability
A sorting algorithm is **stable** if it **preserves the relative original order** of elements with duplicate keys.
- *Example:* If we sort records `[(Alice, 85), (Bob, 85)]` by score, a stable algorithm guarantees that `Alice` appears before `Bob` in the output.
- *Significance:* Essential when performing multi-key sorting (e.g., sorting by timestamp and then by category).

### 1.2 In-Place vs Out-of-Place
- **In-Place ($O(1)$ auxiliary space):** Reorganizes elements directly within the input array with minimal/constant extra memory (e.g., Bubble, Selection, Insertion, Heap Sort).
- **Out-of-Place ($O(n)$ auxiliary space):** Requires allocating secondary buffers or arrays to hold elements during processing (e.g., Merge Sort, Counting Sort).

### 1.3 The Comparison Lower Bound: $\Omega(n \log n)$
For any algorithm based strictly on **pairwise comparisons** ($A[i] < A[j]$), the decision tree must have at least $n!$ leaves (representing all possible permutations of $n$ elements). The minimum tree height is:
$$\log_2(n!) = \Theta(n \log n)$$
Therefore, **no comparison-based sorting algorithm can achieve a worst-case time complexity better than $O(n \log n)$**.

Algorithms such as **Counting Sort** and **Radix Sort** beat this lower bound because they do not rely on element-to-element comparisons; instead, they exploit integer positional arithmetic and digit indexing!

## 2. Environment Setup & Visualization Utilities

Let us build helper utilities to:
1. **Validate** whether an array is properly sorted.
2. **Visualize** array states in real-time using formatted ASCII bars.
3. **Track execution steps** to inspect internal pointer states.

In [ ]:
import time
import random
from typing import List, Callable, Any

def is_sorted(arr: List[Any]) -> bool:
    """Validates if the array is strictly non-decreasing."""
    return all(arr[i] <= arr[i + 1] for i in range(len(arr) - 1))

def print_array_bars(arr: List[int], title: str = "", highlight: List[int] = None) -> None:
    """Renders an array as visual ASCII horizontal bars with optional index highlighting."""
    if title:
        print(f"\n--- {title} ---")
    highlight = highlight or []
    for idx, val in enumerate(arr):
        bar = "█" * (val if val >= 0 else 0)
        marker = "👉 " if idx in highlight else "   "
        print(f"{marker}[{idx:2d}] {val:3d} | {bar}")

# Demonstration test
demo_data = [5, 2, 9, 1, 7]
print_array_bars(demo_data, "Initial Demonstration Array", highlight=[1, 3])
print("Is sorted?", is_sorted(demo_data))

## 3. Elementary Quadratic Algorithms ($O(n^2)$)

---

### 3.1 Bubble Sort (Adjacent Swaps)

#### 💡 How It Works:
Repeatedly iterates through the list, comparing adjacent items `(arr[j], arr[j+1])`. If they are out of order, they are swapped. After each full pass, the largest remaining unsorted element "bubbles up" to its final position at the end of the array.

#### 🚀 Early-Exit Optimization:
If a complete pass completes without a single swap occurring, the array is already 100% sorted! With this `swapped` boolean flag, best-case performance improves from $O(n^2)$ down to **$O(n)$**.

```
Pass 1: [5, 1, 4, 2, 8] -> Compare 5 & 1 -> Swap -> [1, 5, 4, 2, 8]
Pass 2: [1, 5, 4, 2, 8] -> Compare 5 & 4 -> Swap -> [1, 4, 5, 2, 8]
Pass 3: [1, 4, 5, 2, 8] -> Compare 5 & 2 -> Swap -> [1, 4, 2, 5, 8]
Pass 4: [1, 4, 2, 5, 8] -> Compare 5 & 8 -> Keep -> [1, 4, 2, 5, 8] (8 is locked!)
```

In [ ]:
def bubble_sort(arr: List[int], verbose: bool = False) -> List[int]:
    """
    Sorts a list using Bubble Sort with early-exit optimization.
    
    Complexity:
        - Time: Best O(n), Average O(n²), Worst O(n²)
        - Space: O(1) auxiliary (In-place)
        - Stability: Stable
    """
    a = arr.copy()
    n = len(a)
    passes = 0
    swaps_total = 0
    
    for i in range(n):
        swapped = False
        passes += 1
        for j in range(0, n - i - 1):
            if a[j] > a[j + 1]:
                a[j], a[j + 1] = a[j + 1], a[j]
                swapped = True
                swaps_total += 1
                if verbose:
                    print(f"  Pass {i+1}, Swap ({a[j+1]} <-> {a[j]}): {a}")
        if not swapped:
            # Early exit: array is already sorted
            break
            
    if verbose:
        print(f"-> Completed in {passes} passes with {swaps_total} total swaps.")
    return a

# Demonstration
arr_sample = [64, 34, 25, 12, 22, 11, 90]
print("Original:", arr_sample)
sorted_arr = bubble_sort(arr_sample, verbose=True)
print("Sorted  :", sorted_arr)
assert is_sorted(sorted_arr)

---

### 3.2 Selection Sort (Minimum Selection)

#### 💡 How It Works:
Partitions the array conceptually into two segments: the **sorted subarray** (on the left) and the **unsorted subarray** (on the right).
In each iteration, it finds the **minimum element** in the unsorted subarray and swaps it with the first unsorted element.

#### ⚠️ Critical Properties:
- **Fixed Number of Comparisons:** Always performs $\frac{n(n-1)}{2}$ comparisons regardless of initial array ordering. Thus, Best = Average = Worst = $O(n^2)$.
- **Minimal Memory Writes:** Performs at most $n - 1$ swaps ($O(n)$ memory write operations). Highly valuable when writing to memory is physically costly (e.g., Flash/EEPROM memory).
- **Unstable:** Swapping over long distances can violate the relative ordering of duplicate keys.

In [ ]:
def selection_sort(arr: List[int], verbose: bool = False) -> List[int]:
    """
    Sorts a list using Selection Sort.
    
    Complexity:
        - Time: Best O(n²), Average O(n²), Worst O(n²)
        - Space: O(1) auxiliary (In-place)
        - Stability: Unstable (by default)
    """
    a = arr.copy()
    n = len(a)
    
    for i in range(n - 1):
        min_idx = i
        for j in range(i + 1, n):
            if a[j] < a[min_idx]:
                min_idx = j
                
        if min_idx != i:
            a[i], a[min_idx] = a[min_idx], a[i]
            if verbose:
                print(f"  Step {i+1}: Smallest value {a[i]} moved to index {i} -> {a}")
        elif verbose:
            print(f"  Step {i+1}: Index {i} already holds the smallest remaining value ({a[i]})")
            
    return a

arr_sample = [29, 10, 14, 37, 13]
print("Original:", arr_sample)
sorted_arr = selection_sort(arr_sample, verbose=True)
print("Sorted  :", sorted_arr)
assert is_sorted(sorted_arr)

---

### 3.3 Insertion Sort (Incremental Insertion)

#### 💡 How It Works:
Mirrors how most people organize playing cards in their hand:
1. Treats the first element as trivially sorted.
2. Takes the next element (**key**) and compares it backwards (right-to-left) against items in the sorted prefix.
3. Shifts all elements greater than the key one position to the right, then drops the key into its exact position.

#### 🌟 Why Insertion Sort is Essential in Practice:
- **Adaptive $O(n)$ for Nearly-Sorted Data:** If data is already mostly sorted, each item makes $O(1)$ comparisons, yielding lightning-fast **$O(n)$** performance!
- **Extremely Low Constant Factor:** For small collections ($n \le 16$ or $32$), its execution overhead is much lower than complex $O(n \log n)$ algorithms.
- **Engine of Hybrid Algorithms:** Powers the base cases of **Timsort** (standard in Python & Java) and **Introsort** (standard in C++ `std::sort`).

In [ ]:
def insertion_sort(arr: List[int], verbose: bool = False) -> List[int]:
    """
    Sorts a list using Insertion Sort.
    
    Complexity:
        - Time: Best O(n) (nearly sorted), Average O(n²), Worst O(n²)
        - Space: O(1) auxiliary (In-place)
        - Stability: Stable
    """
    a = arr.copy()
    n = len(a)
    
    for i in range(1, n):
        key = a[i]
        j = i - 1
        # Shift elements of a[0..i-1] that are greater than key to the right
        while j >= 0 and a[j] > key:
            a[j + 1] = a[j]
            j -= 1
        a[j + 1] = key
        
        if verbose:
            print(f"  Inserted {key:2d} at index {j+1}: {a}")
            
    return a

arr_sample = [12, 11, 13, 5, 6]
print("Original:", arr_sample)
sorted_arr = insertion_sort(arr_sample, verbose=True)
print("Sorted  :", sorted_arr)
assert is_sorted(sorted_arr)

## 4. Efficient Divide-and-Conquer & Heap Algorithms ($O(n \log n)$)

These algorithms achieve the optimal theoretical comparison-based asymptotic bound.

---

### 4.1 Heap Sort (Binary Max-Heap)

#### 💡 How It Works:
Utilizes a **Max-Heap** data structure (a complete binary tree where each parent node is $\ge$ its children):
1. **Phase 1 (Build Max-Heap):** Transforms the arbitrary array into a Max-Heap in-place in linear time $O(n)$.
2. **Phase 2 (Successive Extraction):** The maximum element is always at root `arr[0]`. Swap `arr[0]` with `arr[end]`, reduce active heap size by 1, and restore the Max-Heap invariant via `heapify` (time $O(\log n)$).
3. Repeat until all elements have been extracted.

#### 📊 Array Indexing in Binary Heaps:
For any node at index $i$:
- Left Child: $2i + 1$
- Right Child: $2i + 2$
- Parent: $\lfloor \frac{i - 1}{2} \rfloor$

#### ⚖️ Advantages & Trade-offs:
- ✅ **Strict $O(n \log n)$ Worst-Case Guarantee** with zero degradation risk (unlike standard QuickSort).
- ✅ **$O(1)$ Auxiliary Space** (completely in-place, unlike MergeSort).
- ❌ **Poor Cache Locality:** Non-sequential array jumps hopping across powers of 2 in memory.

In [ ]:
def heap_sort(arr: List[int], verbose: bool = False) -> List[int]:
    """
    Sorts a list using Heap Sort in-place.
    
    Complexity:
        - Time: Best O(n log n), Average O(n log n), Worst O(n log n)
        - Space: O(1) auxiliary (In-place)
        - Stability: Unstable
    """
    a = arr.copy()
    n = len(a)
    
    def heapify(size: int, root: int):
        """Maintains max-heap property for subtree rooted at 'root'."""
        largest = root
        left = 2 * root + 1
        right = 2 * root + 2
        
        if left < size and a[left] > a[largest]:
            largest = left
        if right < size and a[right] > a[largest]:
            largest = right
            
        if largest != root:
            a[root], a[largest] = a[largest], a[root]
            heapify(size, largest)

    # 1. Build Max-Heap from last non-leaf node up to root
    for i in range(n // 2 - 1, -1, -1):
        heapify(n, i)
        
    if verbose:
        print(f"  Initial Max-Heap built: {a}")

    # 2. Extract elements from heap one by one
    for i in range(n - 1, 0, -1):
        # Move current root (maximum) to the end
        a[0], a[i] = a[i], a[0]
        if verbose:
            print(f"  Extracted {a[i]:2d} to index {i:2d} | Remaining heap: {a[:i]}")
        # Sift down root on reduced heap
        heapify(i, 0)
        
    return a

arr_sample = [12, 11, 13, 5, 6, 7]
print("Original:", arr_sample)
sorted_arr = heap_sort(arr_sample, verbose=True)
print("Sorted  :", sorted_arr)
assert is_sorted(sorted_arr)

---

### 4.2 Merge Sort (Divide and Conquer Interleaving)

#### 💡 How It Works:
Classic **Divide and Conquer** paradigm:
1. **Divide:** Recursively bisects the array into two halves until subarrays have size $\le 1$ (trivially sorted).
2. **Conquer:** Recursively sorts each half.
3. **Combine (Merge):** Merges two sorted subarrays into one sorted array in linear time $O(n)$ using two pointers.

```
Divide:           [38, 27, 43, 3, 9, 82, 10]
             [38, 27, 43, 3]      [9, 82, 10]
          [38, 27]   [43, 3]     [9, 82]   [10]
         [38] [27]  [43]  [3]   [9]  [82]  [10]
Conquer / Merge:
          [27, 38]   [3, 43]     [9, 82]   [10]
             [3, 27, 38, 43]        [9, 10, 82]
Final Result:     [3, 9, 10, 27, 38, 43, 82]
```

#### ⚖️ Advantages & Trade-offs:
- ✅ **Guaranteed Stability:** Preserves initial ordering of equal items with `<=` merge condition.
- ✅ **Ideal for Linked Lists & External Sorting:** Exceptional when dataset is larger than available RAM and resides on disk.
- ❌ **Memory Footprint:** Requires $O(n)$ auxiliary array space to merge elements.

In [ ]:
def merge_sort(arr: List[int], verbose: bool = False, depth: int = 0) -> List[int]:
    """
    Sorts a list using Merge Sort (Top-Down recursive).
    
    Complexity:
        - Time: O(n log n) across all cases (Best, Average, Worst)
        - Space: O(n) auxiliary
        - Stability: Stable
    """
    if len(arr) <= 1:
        return arr
        
    mid = len(arr) // 2
    indent = "  " * depth
    if verbose:
        print(f"{indent}Split: {arr[:mid]} | {arr[mid:]}")
        
    left = merge_sort(arr[:mid], verbose=verbose, depth=depth + 1)
    right = merge_sort(arr[mid:], verbose=verbose, depth=depth + 1)
    
    # Merge phase
    merged = []
    i = j = 0
    while i < len(left) and j < len(right):
        # '<=' maintains stability
        if left[i] <= right[j]:
            merged.append(left[i])
            i += 1
        else:
            merged.append(right[j])
            j += 1
            
    merged.extend(left[i:])
    merged.extend(right[j:])
    
    if verbose:
        print(f"{indent}Merged: {merged}")
    return merged

arr_sample = [38, 27, 43, 3, 9, 82, 10]
print("Original:", arr_sample)
sorted_arr = merge_sort(arr_sample, verbose=True)
print("Sorted  :", sorted_arr)
assert is_sorted(sorted_arr)

---

### 4.3 Quick Sort (Pivot Partitioning)

#### 💡 How It Works:
Also leverages **Divide and Conquer**, performing the bulk of computational work during the **partitioning step**:
1. Selects a **Pivot** element.
2. Reorganizes the array so that all items smaller than the pivot move to the left, and all items greater move to the right. The pivot is now in its final, permanently sorted position!
3. Recursively applies the algorithm to left and right subarrays.

#### 🎯 Partitioning Schemes & Optimizations:
- **Lomuto Partition:** Simple linear single-pointer scan.
- **Hoare Partition:** Two bidirectional pointers converging towards the center, reducing overall swap counts.
- **Randomized Pivot Selection:** Completely avoids the $O(n^2)$ worst-case trap that occurs when selecting fixed endpoints on already sorted/reversed input.

In [ ]:
def quick_sort(arr: List[int], verbose: bool = False) -> List[int]:
    """
    Sorts a list using Quick Sort in-place with randomized pivot selection.
    
    Complexity:
        - Time: Best O(n log n), Average O(n log n), Worst O(n²) (virtually eliminated by random pivot)
        - Space: O(log n) recursion call stack
        - Stability: Unstable
    """
    a = arr.copy()
    
    def _quick_sort(low: int, high: int, depth: int = 0):
        if low < high:
            p_idx = partition(low, high, depth)
            _quick_sort(low, p_idx - 1, depth + 1)
            _quick_sort(p_idx + 1, high, depth + 1)
            
    def partition(low: int, high: int, depth: int) -> int:
        # Choose random pivot and swap with end element
        rand_idx = random.randint(low, high)
        a[rand_idx], a[high] = a[high], a[rand_idx]
        
        pivot = a[high]
        i = low - 1  # Boundary of smaller elements
        
        for j in range(low, high):
            if a[j] <= pivot:
                i += 1
                a[i], a[j] = a[j], a[i]
                
        a[i + 1], a[high] = a[high], a[i + 1]
        
        if verbose:
            indent = "  " * depth
            print(f"{indent}Pivot = {pivot:2d} fixed at index {i+1:2d} -> {a[low:high+1]}")
            
        return i + 1

    if len(a) > 1:
        _quick_sort(0, len(a) - 1)
    return a

arr_sample = [10, 80, 30, 90, 40, 50, 70]
print("Original:", arr_sample)
sorted_arr = quick_sort(arr_sample, verbose=True)
print("Sorted  :", sorted_arr)
assert is_sorted(sorted_arr)

## 5. Non-Comparison Linear Time Algorithms ($O(n + k)$)

These algorithms **do not compare** elements against one another. They harness integer keys and digit arithmetic to break through the theoretical $\Omega(n \log n)$ comparison ceiling.

---

### 5.1 Counting Sort (Frequency Counting & Prefix Sums)

#### 💡 How It Works:
Effective when sorting integers within a bounded range $[\min, \max]$:
1. Determines range $k = \max - \min + 1$.
2. Counts frequencies of each key in a count array `count`.
3. Computes **Prefix Sums (cumulative counts)** so each index represents the final boundary position of that value in the sorted array.
4. Builds the output array by iterating backwards (right-to-left) through the original array, preserving **stability**.

#### ⚖️ Advantages & Limitations:
- ✅ **Strict Linear Time $O(n + k)$**.
- ✅ **Stable** when implemented with prefix sums.
- ❌ **Inefficient for Wide Ranges:** Sorting `[1, 1_000_000_000]` would require allocating $10^9$ array slots for just 2 items.

In [ ]:
def counting_sort(arr: List[int], verbose: bool = False) -> List[int]:
    """
    Sorts a list of integers (including negative numbers) using Stable Counting Sort.
    
    Complexity:
        - Time: O(n + k), where k = max(arr) - min(arr) + 1
        - Space: O(n + k) auxiliary
        - Stability: Stable
    """
    if not arr:
        return []
        
    min_val = min(arr)
    max_val = max(arr)
    k = max_val - min_val + 1
    
    # 1. Frequency count with offset
    count = [0] * k
    for num in arr:
        count[num - min_val] += 1
        
    if verbose:
        print(f"  Frequencies (min={min_val}, max={max_val}, k={k}): {count}")

    # 2. Cumulative prefix sums for target positions
    for i in range(1, k):
        count[i] += count[i - 1]
        
    if verbose:
        print(f"  Prefix Sums (accumulated positions): {count}")

    # 3. Stable output placement via reverse iteration
    output = [0] * len(arr)
    for num in reversed(arr):
        idx = count[num - min_val] - 1
        output[idx] = num
        count[num - min_val] -= 1
        
    return output

arr_sample = [4, -2, 2, 8, 3, 3, 1]
print("Original:", arr_sample)
sorted_arr = counting_sort(arr_sample, verbose=True)
print("Sorted  :", sorted_arr)
assert is_sorted(sorted_arr)

---

### 5.2 Radix Sort (Digit-by-Digit LSD Sorting)

#### 💡 How It Works:
Overcomes Counting Sort's wide-range constraint by breaking numbers down into individual digits:
- Sorts numbers digit by digit, from **Least Significant Digit (LSD)** to Most Significant Digit (MSD).
- Uses an **obligatorily stable subroutine** (such as Counting Sort in base 10 or base 256) at each digit pass.
- Stability guarantees that sorting higher-order digits preserves the relative ordering previously established by lower-order digits!

```
Original      : [170, 045, 075, 090, 802, 024, 002, 066]
Units Digit   : [170, 090, 802, 002, 024, 045, 075, 066]
Tens Digit    : [802, 002, 024, 045, 066, 170, 075, 090]
Hundreds Digit: [002, 024, 045, 066, 075, 090, 170, 802] -> Fully Sorted!
```

#### ⚖️ Complexity & Performance:
- **Time Complexity:** $O(d \cdot (n + k))$, where $d$ is the number of digits in the maximum value and $k$ is the radix base (e.g., $k=10$).
- When $d$ is fixed and small (e.g., 32-bit integers with base 256, where $d = 4$), Radix Sort executes in blazing-fast linear $O(n)$ time.

In [ ]:
def radix_sort(arr: List[int], verbose: bool = False) -> List[int]:
    """
    Sorts a list of integers using Radix Sort LSD (Least Significant Digit).
    Handles positive and negative numbers seamlessly via two partitions.
    
    Complexity:
        - Time: O(d * (n + k)) where d = number of digits, k = base (10)
        - Space: O(n + k) auxiliary
        - Stability: Stable
    """
    if not arr:
        return []
        
    negatives = [-x for x in arr if x < 0]
    positives = [x for x in arr if x >= 0]
    
    def _radix_sort_positive(data: List[int]) -> List[int]:
        if not data:
            return []
        max_num = max(data)
        exp = 1  # 1 for units, 10 for tens, 100 for hundreds...
        
        current = data.copy()
        n = len(current)
        
        while max_num // exp > 0:
            output = [0] * n
            count = [0] * 10
            
            for num in current:
                digit = (num // exp) % 10
                count[digit] += 1
                
            for i in range(1, 10):
                count[i] += count[i - 1]
                
            for num in reversed(current):
                digit = (num // exp) % 10
                idx = count[digit] - 1
                output[idx] = num
                count[digit] -= 1
                
            current = output
            if verbose:
                print(f"  Pass with exp {exp:4d} -> {current}")
            exp *= 10
            
        return current

    sorted_pos = _radix_sort_positive(positives)
    sorted_neg = [-x for x in reversed(_radix_sort_positive(negatives))]
    
    return sorted_neg + sorted_pos

arr_sample = [170, 45, 75, 90, 802, 24, 2, 66]
print("Original:", arr_sample)
sorted_arr = radix_sort(arr_sample, verbose=True)
print("Sorted  :", sorted_arr)
assert is_sorted(sorted_arr)

## 6. Benchmarking Lab & Real-World Performance Analysis

Let us pit all 8 sorting algorithms head-to-head across 4 distinct distribution scenarios:
1. **Uniform Random:** Standard random distribution.
2. **Nearly Sorted:** Sorted array with ~5% noise/perturbations (the domain where Insertion Sort thrives).
3. **Reversed:** Inversely ordered elements (worst-case stress test).
4. **Few Unique Keys:** Only 5 unique numbers duplicated across the entire array.

In [ ]:
# Unified registry of all 8 algorithms
ALGORITHMS: dict[str, Callable[[List[int]], List[int]]] = {
    "Bubble Sort": bubble_sort,
    "Selection Sort": selection_sort,
    "Insertion Sort": insertion_sort,
    "Heap Sort": heap_sort,
    "Merge Sort": merge_sort,
    "Quick Sort": quick_sort,
    "Counting Sort": counting_sort,
    "Radix Sort": radix_sort,
    "Python Timsort (built-in)": sorted
}

def benchmark_algorithm(sort_fn: Callable, data: List[int], runs: int = 3) -> float:
    """Runs multiple passes and returns the average execution time in milliseconds."""
    total_time = 0.0
    for _ in range(runs):
        data_copy = data.copy()
        t0 = time.perf_counter()
        res = sort_fn(data_copy)
        t1 = time.perf_counter()
        assert is_sorted(res), f"Sorting verification failed for {sort_fn.__name__}!"
        total_time += (t1 - t0)
    return (total_time / runs) * 1000.0  # Milliseconds

# Dataset Generation (N = 1000 items)
N = 1000
random.seed(42)

datasets = {
    "1. Random": [random.randint(0, 5000) for _ in range(N)],
    "2. Nearly Sorted": [i if random.random() > 0.05 else random.randint(0, N) for i in range(N)],
    "3. Reversed": list(range(N, 0, -1)),
    "4. Few Unique": [random.choice([10, 20, 30, 40, 50]) for _ in range(N)]
}

# Benchmark Results Matrix Table
print(f"{'Algorithm':<26} | {'Random (ms)':<15} | {'Nearly Sorted':<15} | {'Reversed (ms)':<15} | {'Few Unique':<15}")
print("-" * 96)

for name, fn in ALGORITHMS.items():
    times = []
    for d_name, d_list in datasets.items():
        t_ms = benchmark_algorithm(fn, d_list)
        times.append(f"{t_ms:9.3f} ms")
    print(f"{name:<26} | {times[0]:<15} | {times[1]:<15} | {times[2]:<15} | {times[3]:<15}")

## 7. Master Complexity Matrix & Production Decision Guide

### 📊 Master Complexity Matrix

| Algorithm | Best Case | Average Case | Worst Case | Auxiliary Space | Stability | Mechanism |
| :--- | :---: | :---: | :---: | :---: | :---: | :--- |
| **Bubble Sort** | $O(n)$ | $O(n^2)$ | $O(n^2)$ | $O(1)$ | ✅ Stable | Adjacent Swapping |
| **Selection Sort** | $O(n^2)$ | $O(n^2)$ | $O(n^2)$ | $O(1)$ | ❌ Unstable | Minimum Selection |
| **Insertion Sort** | $O(n)$ | $O(n^2)$ | $O(n^2)$ | $O(1)$ | ✅ Stable | Incremental Insertion |
| **Heap Sort** | $O(n \log n)$ | $O(n \log n)$ | $O(n \log n)$ | $O(1)$ | ❌ Unstable | Heap Selection & Sift-Down |
| **Merge Sort** | $O(n \log n)$ | $O(n \log n)$ | $O(n \log n)$ | $O(n)$ | ✅ Stable | Divide and Conquer |
| **Quick Sort** | $O(n \log n)$ | $O(n \log n)$ | $O(n^2)$ | $O(\log n)$ | ❌ Unstable | Pivot Partitioning |
| **Counting Sort** | $O(n + k)$ | $O(n + k)$ | $O(n + k)$ | $O(n + k)$ | ✅ Stable | Non-Comparison Frequency Map |
| **Radix Sort** | $O(d \cdot (n+k))$ | $O(d \cdot (n+k))$ | $O(d \cdot (n+k))$ | $O(n + k)$ | ✅ Stable | Non-Comparison Digit Bucketing |

---

### 🎯 Production Decision Framework: Which to Choose?

1. **Small Data ($n < 30$) or Nearly 100% Sorted Streams:**
   - 👉 Use **Insertion Sort** (minimal constant overhead, zero allocations, $O(n)$ best-case).
2. **General-Purpose In-Memory Sorting (High Cache Locality):**
   - 👉 Use **QuickSort** (or language native **Introsort** / **Timsort**).
3. **Embedded Systems / Memory-Constrained Environments with Strict Real-Time Deadlines:**
   - 👉 Use **HeapSort** (guaranteed $O(n \log n)$ with zero worst-case penalty and $O(1)$ RAM).
4. **Stability Requirement or External Sorting (Massive disk files):**
   - 👉 Use **MergeSort** (predictable, stable, and accesses disk blocks sequentially).
5. **Bounded Small Integer Ranges (e.g., ages 0-120, exam scores 0-100):**
   - 👉 Use **Counting Sort** (unbeatable $O(n)$ linear speed).
6. **Fixed-Width Uniform Keys (e.g., ZIP codes, IDs, 32-bit hashes):**
   - 👉 Use **Radix Sort** (linear time without pairwise comparisons).

---

<div align="center">

### 🏆 End of Sorting Algorithms Masterclass!
Run the cells, inspect the step traces, and experiment with different dataset variations!

</div>